In [12]:
%pip install pandas numpy statsmodels


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# loading csv
df = pd.read_csv("../data/train.csv")



C:\Users\perez\AppData\Local\Temp\ipykernel_17744\4285296687.py:6: DtypeWarning: Columns (11: Grid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/train.csv")


In [14]:
# cleaning whitespace from text columns
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

# filtering
cols = [
    "Priority",
    "InitialCaseTypeGroup",
    "CallReceivedType",
    "OccurrenceHour",
    "OccurrenceWeekday",
    "OccurrenceMonth",
    "Zone",
]
df_clean = df[df["Priority"] != 9].dropna(subset=cols).copy()

# dropping rare categories to prevent errors we can add them in later if yall think we need them and deal with issues
for col in ["InitialCaseTypeGroup", "CallReceivedType", "Zone"]:
    counts = df_clean[col].value_counts()
    valid_categories = counts[counts >= 15].index
    df_clean = df_clean[df_clean[col].isin(valid_categories)]



In [15]:
# creating binary dependent variable (1 and 2 = high priority, 0 = low)
df_clean["high_priority"] = np.where(df_clean["Priority"] <= 2, 1, 0)

print(f"n rows fitted      : {len(df_clean):,}")
print(f"high_priority rate : {df_clean['high_priority'].mean():.3f}")




n rows fitted      : 380,871
high_priority rate : 0.703


In [16]:
# fitting logistic regression model
# C() tells statsmodels to treat categorical text/number codes as dummy variables
formula = """
high_priority ~ C(InitialCaseTypeGroup) 
              + C(CallReceivedType) 
              + OccurrenceHour 
              + C(OccurrenceWeekday) 
              + C(OccurrenceMonth) 
              + C(Zone)
"""

model = smf.logit(formula, data=df_clean).fit(disp=False)

# statistical summary
print()
print(model.summary())




                           Logit Regression Results                           
Dep. Variable:          high_priority   No. Observations:               380871
Model:                          Logit   Df Residuals:                   380818
Method:                           MLE   Df Model:                           52
Date:                Wed, 18 Nov 2026   Pseudo R-squ.:                  0.2479
Time:                        20:44:52   Log-Likelihood:            -1.7420e+05
converged:                       True   LL-Null:                   -2.3163e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------------
Intercept                                                     1.0188      0.029     35.519      0.000       0.963  

In [17]:
# odds ratios (OR) & 95% confidence intervals
print("\nOdds ratios with 95% confidence intervals:")
coef = model.params
ci = model.conf_int()
or_df = pd.DataFrame({
    "OR": np.exp(coef),
    "OR_lo": np.exp(ci[0]),
    "OR_hi": np.exp(ci[1]),
    "p_value": model.pvalues
})
print(or_df.round(3))




Odds ratios with 95% confidence intervals:
                                                        OR   OR_lo   OR_hi  \
Intercept                                            2.770   2.618   2.930   
C(InitialCaseTypeGroup)[T.Assault/Homicide/Sex ...   5.141   4.877   5.418   
C(InitialCaseTypeGroup)[T.Code Violation]            0.759   0.726   0.795   
C(InitialCaseTypeGroup)[T.Domestic]                  3.565   3.426   3.711   
C(InitialCaseTypeGroup)[T.Drugs/Alcohol]             1.387   1.313   1.466   
C(InitialCaseTypeGroup)[T.Fire]                     46.729  32.425  67.342   
C(InitialCaseTypeGroup)[T.Fraud]                     0.157   0.133   0.186   
C(InitialCaseTypeGroup)[T.Health and Welfare]        8.306   7.997   8.628   
C(InitialCaseTypeGroup)[T.Juvenile]                  5.726   5.265   6.228   
C(InitialCaseTypeGroup)[T.Other]                     8.190   7.950   8.437   
C(InitialCaseTypeGroup)[T.Property]                  1.929   1.869   1.992   
C(InitialCaseTypeGro